In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Imports

In [49]:
! pip install -q chonkie sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 3.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.2/387.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 74.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.0 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have n

In [50]:
from kaggle_secrets import UserSecretsClient
import wandb

#from chonkie import RecursiveChunker

import numpy as np              
import pandas as pd             
import matplotlib.pyplot as plt 
import seaborn as sns           
import torch                    
import torch.nn as nn           
from collections import Counter 
from string import punctuation  
import warnings                 
import string
import re

from transformers import pipeline,AutoTokenizer,AutoModel,AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, CrossEncoder 
import faiss 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

from tqdm import tqdm

In [17]:
%matplotlib inline
plt.style.use('fivethirtyeight')
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

print(f"PyTorch Version: {torch.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42)
np.random.seed(42)

PyTorch Version: 2.10.0+cu128
NumPy Version: 2.4.6
Pandas Version: 2.3.3
CUDA Available: True
CUDA Version: 12.8
GPU Device: Tesla T4


# W&B

In [18]:
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("wandb_api")

In [19]:
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [20]:
CONFIG={
    "project_name":"23f2000391-t22026",
    "embedding_model":"all-MiniLM-L6-v2",
    "knowledge_base":"train.csv",
    "search_index":"IndexFlatL2",
    "zs_model":"bart-large-mnli",
    "cross_encoder":"ms-marco-MiniLM-L-6-v2",
    "retrieval_k":5,
    "split":"Stratified-K-Fold",
    "folds_num":5
}

In [21]:
wandb.init(
    project=CONFIG["project_name"],
    name='vanilla-RAG-system',
    config=CONFIG
)

# Dataset

In [22]:
train=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [23]:
train.drop(columns='id',inplace=True)
train.head()

,prompt,A,B,C,D,E,answer
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [24]:
test=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [25]:
test.drop(columns='id',inplace=True)
test.head()

,prompt,A,B,C,D,E
0,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


## EDA

In [26]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   prompt  2000 non-null   object
 1   A       2000 non-null   object
 2   B       2000 non-null   object
 3   C       2000 non-null   object
 4   D       2000 non-null   object
 5   E       2000 non-null   object
 6   answer  2000 non-null   object
dtypes: object(7)
memory usage: 109.5+ KB


In [27]:
train.describe()

,prompt,A,B,C,D,E,answer
count,2000,2000,2000,2000,2000,2000,2000
unique,1758,316,328,303,318,320,5
top,Choose the correct answer: What is the main se...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,An improper rotation is the combination of a r...,B
freq,4,21,21,21,21,21,490


There might be some duplicates since only 1758 out of the 2000 prompts are unique

In [28]:
print(f"Number of Duplicates: {train.duplicated().sum()}")
print("Shape before dropping duplicates",train.shape)
train.drop_duplicates(inplace=True)
print("Shape after dropping duplicates",train.shape)

Number of Duplicates: 183
Shape before dropping duplicates (2000, 7)
Shape after dropping duplicates (1817, 7)


In [29]:
train['answer'].value_counts()

answer
B    442
C    423
D    329
A    328
E    295
Name: count, dtype: int64

In [30]:
prompt_word_length=train['prompt'].apply(lambda x:len(x.split()))
'''print(f"Avg Prompt Words: {prompt_word_length.mean()}")
print(f"Max Prompt Words: {prompt_word_length.max()}")
print(f"Min Prompt Words: {prompt_word_length.min()}")'''
prompt_word_length.describe()

count    1817.000000
mean       18.057237
std         6.844080
min         3.000000
25%        14.000000
50%        17.000000
75%        22.000000
max        51.000000
Name: prompt, dtype: float64

## Preprocessing

In [31]:
train['clean_prompt']=train['prompt'].str.lower().apply(lambda x: str(x).translate(str.maketrans("", "", string.punctuation)))
train.head()

,prompt,A,B,C,D,E,answer,clean_prompt
0,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...
1,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is acceleratorbased lightion fusion
2,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...
3,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...
4,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...


## Corpus for vocabulary

In [32]:
text=' '.join(train['clean_prompt'].tolist())
words=text.split()
print("Total words in corpus:",len(words))
assert(prompt_word_length.sum()==len(words))

Total words in corpus: 32810


In [33]:
word_freq=Counter(words)
print(f"Most Frequent Words:")
for i,j in (word_freq.most_common(10)):
    print(f"{i}: {j}")
print(f"\nLeast Frequent Words:")
for i,j in (word_freq.most_common()[-11:-1]):
    print(f"{i}: {j}")

Most Frequent Words:
the: 4989
is: 1820
what: 1641
of: 1365
correct: 1052
following: 671
in: 653
option: 549
answer: 545
and: 407

Least Frequent Words:
disparity: 3
presence: 3
antimatter: 3
observable: 3
obtaining: 3
surgical: 3
resection: 3
specimens: 3
essential: 2
significant: 2


In [34]:
word_freq_sorted=word_freq.most_common()
vocab_to_int={word: idx+1 for idx, (word, _) in enumerate(word_freq_sorted)}

## Split

In [35]:
#train_df,val_df=train_test_split(train,test_size=0.2,stratify=train["answer"],random_state=42)

skf=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

#print(train_df["answer"].value_counts()/len(train_df))
#print(val_df["answer"].value_counts()/len(val_df))

# Scoring

In [36]:
def map_at_3(true,pred):
    scores=[]
    for actual,preds in zip(true,pred):
        score=0.0
        for rank,pred in enumerate(preds,start=1):
            if pred==actual:
                score=1.0/rank
                break
        scores.append(score)
    return np.mean(scores)

In [37]:
def top3_accuracy(y_true, predictions):
    return np.mean([truth in pred for truth, pred in zip(y_true, predictions)])

In [38]:
def top1_accuracy(y_true, predictions):
    top1=[pred[0] if len(pred) else "Z" for pred in predictions]

    return accuracy_score(y_true,top1)

# PreTrained Models

## Zero-Shot Classification

In [39]:
'''qwen_model_name="Qwen/Qwen2.5-7B-Instruct"
qwen_tokenizer=AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model=AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    dtype=torch.float16,
    device_map="auto"
)'''

'qwen_model_name="Qwen/Qwen2.5-7B-Instruct"\nqwen_tokenizer=AutoTokenizer.from_pretrained(qwen_model_name)\nqwen_model=AutoModelForCausalLM.from_pretrained(\n    qwen_model_name,\n    dtype=torch.float16,\n    device_map="auto"\n)'

In [40]:
'''def create_zero_shot_prompt(row):
    zero_shot_prompt=f"""
    You are solving a multiple choice question containing 5 choices.
    Question:
    {row["prompt"]}
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    Return ONLY the three most likely answer labels with a single space separating them.
    Example:
    C A D
    """
    return zero_shot_prompt'''

'def create_zero_shot_prompt(row):\n    zero_shot_prompt=f"""\n    You are solving a multiple choice question containing 5 choices.\n    Question:\n    {row["prompt"]}\n    Choices:\n    A. {row["A"]}\n    B. {row["B"]}\n    C. {row["C"]}\n    D. {row["D"]}\n    E. {row["E"]}\n    Return ONLY the three most likely answer labels with a single space separating them.\n    Example:\n    C A D\n    """\n    return zero_shot_prompt'

In [41]:
'''def create_few_shot_prompt(row):
    few_shot_examples = []
    for label in ["A", "B", "C", "D", "E"]:
        example = train_df[train_df["answer"] == label].sample(n=1,random_state=42).iloc[0]
        few_shot_examples.append(example)
    prompt = """You are an expert at solving multiple-choice questions.
                Below are some solved examples.\n"""

    for i, ex in enumerate(few_shot_examples, 1):

        prompt += f"""Example {i}
        Question: {ex["prompt"]}
        
        Choices:
        A. {ex["A"]}
        B. {ex["B"]}
        C. {ex["C"]}
        D. {ex["D"]}
        E. {ex["E"]}
        
        Correct Answer:
        {ex["answer"]}
        
        """
        
    prompt += f"""
    Now answer the following question.
    
    Question:
    {row["prompt"]}
    
    Choices:
    A. {row["A"]}
    B. {row["B"]}
    C. {row["C"]}
    D. {row["D"]}
    E. {row["E"]}
    
    Rank the choices based on their probability of being the correct answer.
    Then return the top three answer labels separated by spaces.
    
    Example output:
    C A D
    
    Do not explain your answer.
    """

    return prompt'''

'def create_few_shot_prompt(row):\n    few_shot_examples = []\n    for label in ["A", "B", "C", "D", "E"]:\n        example = train_df[train_df["answer"] == label].sample(n=1,random_state=42).iloc[0]\n        few_shot_examples.append(example)\n    prompt = """You are an expert at solving multiple-choice questions.\n                Below are some solved examples.\n"""\n\n    for i, ex in enumerate(few_shot_examples, 1):\n\n        prompt += f"""Example {i}\n        Question: {ex["prompt"]}\n        \n        Choices:\n        A. {ex["A"]}\n        B. {ex["B"]}\n        C. {ex["C"]}\n        D. {ex["D"]}\n        E. {ex["E"]}\n        \n        Correct Answer:\n        {ex["answer"]}\n        \n        """\n        \n    prompt += f"""\n    Now answer the following question.\n    \n    Question:\n    {row["prompt"]}\n    \n    Choices:\n    A. {row["A"]}\n    B. {row["B"]}\n    C. {row["C"]}\n    D. {row["D"]}\n    E. {row["E"]}\n    \n    Rank the choices based on their probability of b

In [42]:
'''def predict_qwen(row):

    prompt = create_few_shot_prompt(row)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_tokenizer(
        text,
        return_tensors="pt",
        truncation=True
    ).to(qwen_model.device)

    with torch.no_grad():

        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    prediction = qwen_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    labels = re.findall(r"\b[A-E]\b", prediction)

    return labels[:3]'''

'def predict_qwen(row):\n\n    prompt = create_few_shot_prompt(row)\n\n    messages = [\n        {\n            "role": "user",\n            "content": prompt\n        }\n    ]\n\n    text = qwen_tokenizer.apply_chat_template(\n        messages,\n        tokenize=False,\n        add_generation_prompt=True\n    )\n\n    inputs = qwen_tokenizer(\n        text,\n        return_tensors="pt",\n        truncation=True\n    ).to(qwen_model.device)\n\n    with torch.no_grad():\n\n        outputs = qwen_model.generate(\n            **inputs,\n            max_new_tokens=5,\n            do_sample=False,\n            pad_token_id=qwen_tokenizer.eos_token_id\n        )\n\n    generated = outputs[0][inputs.input_ids.shape[1]:]\n\n    prediction = qwen_tokenizer.decode(\n        generated,\n        skip_special_tokens=True\n    ).strip()\n\n    labels = re.findall(r"\x08[A-E]\x08", prediction)\n\n    return labels[:3]'

In [43]:
'''predictions = []

fold_map = []
fold_acc = []
fold_top3 = []

for fold,(train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        pred = predict_qwen(row)
        predictions.append(pred)
        torch.cuda.empty_cache()
    map3 = map_at_3(
        val_df["answer"],
        predictions
    )
    
    acc = top1_accuracy(
        val_df["answer"],
        predictions
    )
    
    top3 = top3_accuracy(
        val_df["answer"],
        predictions
    )
    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Accuracy": acc,
        "Top3 Accuracy": top3
    })
    fold_map.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)
print(f"MAP@3 : {np.mean(fold_map):.4f}")
print(f"Accuracy : {np.mean(fold_acc):.4f}")
print(f"Top3 Accuracy : {np.mean(fold_top3):.4f}")'''

'predictions = []\n\nfold_map = []\nfold_acc = []\nfold_top3 = []\n\nfor fold,(train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):\n    train_df=train.iloc[train_idx].reset_index(drop=True)\n    val_df=train.iloc[val_idx].reset_index(drop=True)\n    predictions=[]\n    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):\n        pred = predict_qwen(row)\n        predictions.append(pred)\n        torch.cuda.empty_cache()\n    map3 = map_at_3(\n        val_df["answer"],\n        predictions\n    )\n    \n    acc = top1_accuracy(\n        val_df["answer"],\n        predictions\n    )\n    \n    top3 = top3_accuracy(\n        val_df["answer"],\n        predictions\n    )\n    wandb.log({\n        "Fold": fold + 1,\n        "MAP@3": map3,\n        "Accuracy": acc,\n        "Top3 Accuracy": top3\n    })\n    fold_map.append(map3)\n    fold_acc.append(acc)\n    fold_top3.append(top3)\nprint(f"MAP@3 : {np.mean(fold_map):.4f}")\nprint(f"Accuracy : {np.mean(fold_acc):.4f}"

In [44]:
'''wandb.log({
    "Average MAP@3": np.mean(fold_map),
    "Average Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()'''

'wandb.log({\n    "Average MAP@3": np.mean(fold_map),\n    "Average Accuracy": np.mean(fold_acc),\n    "Average Top3 Accuracy": np.mean(fold_top3)\n})\n\nwandb.finish()'

# Model of Choice

## Basic RAG System

In [51]:
model = SentenceTransformer('all-MiniLM-L6-v2') 
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [54]:
def build_kb(df):
     
    for idx, row in train_df.iterrows(): 
        correct_letter=row['answer'] 
        kb.append(str(row[correct_letter])) 
    kb_embeddings=model.encode(kb, show_progress_bar=False) 
    index=faiss.IndexFlatL2(kb_embeddings.shape[1]) 
    index.add(kb_embeddings)

    return kb,index

In [61]:
def rag_predict(row,kb,index):
    prompt_row=row['prompt']
    
    row_embeddings=np.expand_dims(model.encode(prompt_row, show_progress_bar=False),axis=0)
    row_docs=index.search(row_embeddings, k=5)[1][0]

    docs_5=[kb[i] for i in row_docs] 
    pairs=[[prompt_row, doc] for doc in docs_5] 
    ce_scores=cross_encoder.predict(pairs) 

    best_doc=kb[np.argsort(-ce_scores)[0]]
    aug_string=f"Context: {best_doc} Question: {prompt_row}"
    choices=["A","B","C","D","E"]
    row_labels=[row[i] for i in choices]
    
    zs_pred=zs(aug_string,candidate_labels=row_labels)
    zs_labels=zs_pred['labels'][:3]
    
    top3_preds=[]
    for i in zs_labels:
        top3_preds.append(choices[row_labels.index(i)])

    return top3_preds

In [57]:
fold_map3=[]
fold_acc=[]
fold_top3=[]

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train["answer"])):
    train_df=train.iloc[train_idx].reset_index(drop=True)
    val_df=train.iloc[val_idx].reset_index(drop=True)
    
    print("Fold:",fold+1)
    
    kb,index=build_kb(train_df)

    predictions=[]
    for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
        top3_preds=rag_predict(row,kb,index)        
        predictions.append(top3_preds)
       
    map3=map_at_3(val_df["answer"],predictions)
    acc=top1_accuracy(val_df["answer"],predictions)
    top3=top3_accuracy(val_df["answer"],predictions)

    print(f"MAP@3: {map3:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Top3 Accuracy: {top3:.4f}")

    fold_map3.append(map3)
    fold_acc.append(acc)
    fold_top3.append(top3)

    wandb.log({
        "Fold": fold + 1,
        "MAP@3": map3,
        "Top1 Accuracy": acc,
        "Top3 Accuracy": top3
    })


Fold: 1


100%|██████████| 364/364 [01:12<00:00,  4.99it/s]


MAP@3: 0.5829
Accuracy: 0.3901
Top3 Accuracy: 0.8352
Fold: 2


100%|██████████| 364/364 [01:13<00:00,  4.98it/s]


MAP@3: 0.5472
Accuracy: 0.3434
Top3 Accuracy: 0.8077
Fold: 3


...

MAP@3: 0.5202
Accuracy: 0.3251
Top3 Accuracy: 0.7851
Fold: 4


100%|██████████| 363/363 [01:13<00:00,  4.97it/s]


MAP@3: 0.5615
Accuracy: 0.3747
Top3 Accuracy: 0.7961
Fold: 5


100%|██████████| 363/363 [01:11<00:00,  5.05it/s]

MAP@3: 0.5500
Accuracy: 0.3554
Top3 Accuracy: 0.8182


In [58]:
print("Average MAP@3:", np.mean(fold_map3))
print("Average Accuracy:", np.mean(fold_acc))
print("Average Top3 Accuracy:", np.mean(fold_top3))

wandb.log({
    "Average MAP@3": np.mean(fold_map3),
    "Average Top1 Accuracy": np.mean(fold_acc),
    "Average Top3 Accuracy": np.mean(fold_top3)
})

wandb.finish()

Average MAP@3: 0.5523617796345068
Average Accuracy: 0.35772258044985317
Average Top3 Accuracy: 0.808461235733963


Average MAP@3,▁
Average Top1 Accuracy,▁
Average Top3 Accuracy,▁
Fold,▁▃▅▆█
MAP@3,█▄▁▆▄
Top1 Accuracy,█▃▁▆▄
Top3 Accuracy,█▄▁▃▆
Average MAP@3,0.55236
Average Top1 Accuracy,0.35772
Average Top3 Accuracy,0.80846
Fold,5


# Submission

In [62]:
test_predictions=[]
kb_train,index_train=build_kb(train)
for _, row in tqdm(test.iterrows(), total=len(test)):
    test_predictions.append(rag_predict(row,kb_train,index_train))
test_predictions[:5]

100%|██████████| 500/500 [01:39<00:00,  5.01it/s]


[['D', 'E', 'B'],
 ['E', 'B', 'D'],
 ['D', 'B', 'A'],
 ['E', 'C', 'B'],
 ['D', 'C', 'B']]

In [63]:
submission_preds = [" ".join(pred) for pred in test_predictions]
sub=pd.read_csv(f"/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sub.set_index('ID',inplace=True)
sub.head()

,Prediction
ID,
1,A B C
2,A B C
3,A B C
4,A B C
5,A B C


In [64]:
sub['Prediction']=submission_preds
sub.head()

,Prediction
ID,
1,D E B
2,E B D
3,D B A
4,E C B
5,D C B


In [65]:
sub.to_csv("submission.csv")
print("Submission File Created")

Submission File Created
